In [9]:
import pdfplumber
from pathlib import Path
from pdfplumber.utils.text import extract_text
import re
import json
import pandas as pd

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()
FORMULATION_PATH = Path("Formulación")

In [3]:
file = "Formulación/2020 - Formulación.pdf"

In [4]:
def extraer_nombres_formulas(statement):
    patron = re.compile(r"R\s*E\s*S\s*O\s*L\s*U\s*C\s*I\s*[ÓO]\s*N", re.IGNORECASE)
    m = patron.search(statement)
    end_position = m.start() if m else -1
    compounds_end_position = statement.lower().find("d)")

    # extraer nombre examen
    statement = statement[:end_position]
    lines = statement.strip().split("\n")
    exam_name = lines[-1].lower()
    exam_name = re.sub(r"ejercicio \w+", "", exam_name)

    # compuestos a nombrar
    compounds = statement[:compounds_end_position]
    compounds = compounds.replace("Formule o nombre los compuestos siguientes: ", "").strip()
    return exam_name, compounds

In [5]:
formulation_statements = {}
for file in FORMULATION_PATH.iterdir():
    if file.is_file():
        with pdfplumber.open(file) as pdf:
            year = int(file.name[:4])
            formulation_statements[year] = []
            for page in pdf.pages[1:]:
                statement = page.extract_text()
                formulation_statements[year].append(extraer_nombres_formulas(statement))

In [6]:
formulation_statements

{2003: [('química. 2003. junio. . opción a',
   'a) Peróxido de bario; b) Ácido clórico; c) 1,2-\nEtanodiol;'),
  ('química. 2003. junio. . opción b',
   'a) Hidróxido de plata ; b) Fluoruro de hidrógeno\nc) Etanoamida ;'),
  ('química. 2003. reserva 1. . opción a',
   'a) Sulfito de calcio; b) Hidróxido de estroncio;\nc) Metanal;'),
  ('química. 2003. reserva 1. . opción b',
   'a) Ácido perclórico; b) Óxido de titanio(IV) ;\nc) Fenol ;'),
  ('química. 2003. reserva 2. . opción a',
   'a) Óxido de zirconio(IV); b) Sulfuro de\narsénico(III); c) 3-Metilpentano ;'),
  ('química. 2003. reserva 2. . opción b',
   'a) Hidróxido de magnesio; b) Yodato de potasio;\nc) Etilmetil éter;'),
  ('química. 2003. reserva 3. . opción a',
   'a) Bromuro de hidrógeno; b) Fosfato de litio;\nc) 2-Buteno;'),
  ('química. 2003. reserva 3. . opción b',
   'a) Hidrogenocarbonato de cesio; b) Óxido de\ncadmio; c) o-Dimetilbenceno;'),
  ('química. 2003. reserva 4. . opción a',
   'a) Cromato de calcio; b) Peróx

In [7]:
form_emestrada = []
for year, exercises in formulation_statements.items():
    for ex in exercises:
        form_emestrada.append({"year" : year, "exam" : ex[0], "statement" : ex[1]})

In [11]:
df_emestrada = pd.DataFrame(form_emestrada)

Limpiar texto de `exam`

In [14]:
df_emestrada.exam.head()

0        química. 2003. junio. . opción a
1        química. 2003. junio. . opción b
2    química. 2003. reserva 1. . opción a
3    química. 2003. reserva 1. . opción b
4    química. 2003. reserva 2. . opción a
Name: exam, dtype: str

In [28]:
def format_exam(exam):
    exam = re.sub(r"\s+", " ", exam).replace(".", "")
    exam_parts = exam.split(" ")
    exam_parts = [i.capitalize() for i in exam_parts]

    exam = exam_parts[0] + " - " + ", ".join(exam_parts[1:-2]) + " ".join(exam_parts[-2:])

    return exam

In [30]:
df_emestrada["exam"] = df_emestrada.exam.apply(format_exam)

In [31]:
df_emestrada.head()

,year,exam,statement
0,2003,"Química - 2003, Junio, Opción A","a) Peróxido de bario; b) Ácido clórico; c) 1,2..."
1,2003,"Química - 2003, Junio, Opción B",a) Hidróxido de plata ; b) Fluoruro de hidróge...
2,2003,"Química - 2003, Reserva, 1, Opción A",a) Sulfito de calcio; b) Hidróxido de estronci...
3,2003,"Química - 2003, Reserva, 1, Opción B",a) Ácido perclórico; b) Óxido de titanio(IV) ;...
4,2003,"Química - 2003, Reserva, 2, Opción A",a) Óxido de zirconio(IV); b) Sulfuro de\narsén...


Limpiar texto de `statement`

In [ ]:
for idx, st in df_emestrada.dropna().sample(3).apply(limpiar_formulacion)iterrows():
    print(f"{st['statement']}\n\n")

Formule o nombre los siguientes compuestos: a) Dióxido de paladio; b) Hidrogenosulfato de
sodio; c) 1,2-Dicloropropano;


a) Nitrito de hierro(II); b) Hidruro de berilio;
c) Trimetilamina;


a) Hexafluoruro de azufre; b) Hidrogenofosfato
de potasio; c) Hexan-2-amina;




In [52]:
def limpiar_formulacion(texto):
    if not pd.isna(texto):
        texto = texto.strip().replace(";", "").replace("\n"," ")
        texto = texto.replace("Formule o nombre los siguientes compuestos: ", "")
        return texto
    else:
        return texto

In [57]:
for i in df_emestrada.loc[~df_emestrada.statement.isna(), "statement"].sample(5).apply(limpiar_formulacion):
    print(i)

a) Ácido sulfuroso b) Hidróxido de cobre(II) c) 3,3-Dimetilciclopenteno
a) Hidrogenocarbonato de sodio b) Hidróxido de cobre(I) c) Pent-1-ino
a) Nitrato de hierro(II) b) Cromato de potasio c) Ácido but-3-enoico
a) Clorato de cobalto(III) b) Sulfuro de cinc c) Etanoato de etilo
a) Permanganato de potasio b) Seleniuro de plata c) Butilamina


In [58]:
df_emestrada.loc[:, "statement"] = df_emestrada.loc[:, "statement"].apply(limpiar_formulacion)

In [59]:
df_emestrada.sample(3)

,year,exam,statement
193,2019,"Química - 2019, Reserva, 1, Opción B",a) Hidruro de berilio b) Hidróxido de hierro(I...
21,2015,"Química - 2015, Reserva, 4, Opción B",a) Cloruro de calcio b) Óxido de níquel(II) c)...
24,2016,"Química - 2016, Junio, Opción A",a) Hidróxido de niquel(III) b) Ácido peryódico...


In [62]:
df_emestrada.to_csv("form_emestrada.csv", index = False, encoding="utf-8")

In [63]:
df_emestrada.year.aggregate(["min", "max"])

min    2000
max    2025
Name: year, dtype: int64